# Relative Flux Light Curves using psfScience (raw science flux)

## Motivation

In the DIA (Difference Image Analysis) pipeline:

```
psfFlux  =  psfScience  -  psfTemplate
```

If the template (`psfTemplate`) is incorrect — wrong epoch, template noise, residual
gradients, or detector-dependent biases — then `psfFlux` will carry spurious
variations that do **not** originate from the source itself.

This notebook investigates whether the light-curve variability seen in `psfFlux`
for stable Gaia stars is driven by the **science image flux** (`psfScience`) or by
a bad **template** (`psfTemplate`).

### Strategy

1. **Detect** which science-flux columns are present in the enriched alert table
   (the Fink API names vary: `r:psfScience`, `r:scienceFlux`, etc.).
2. **Compute** relative flux = `psfScience(t) / median(psfScience)` per band,
   exactly as done for `psfFlux` in `04_relativeFlux.ipynb`.
3. **Overlay** `psfScience/median` and `psfFlux/median` on the same axes so the
   contribution of the template can be assessed visually.
4. **Compute** the implied template flux:
   `psfTemplate_est = psfScience - psfFlux`
   and plot its relative variability to check for temporal drifts.

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Created from:** `04_relativeFlux.ipynb`  
- **Last update:** 2026-04-06  
- **Subject:** Fink alert broker — Rubin LSST photometric calibration check via psfScience

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_06"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"  # same input as notebook 04
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Source files (butler preferred; fallback to consdb) ───────────────────────
FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")

# ── Number of top-ranked DIA objects to plot ──────────────────────────────────
N_TOP = 20  # <── change here

# ── Band definitions ──────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Fixed column names ────────────────────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"  # difference-image PSF flux
COL_PSFERR = "r:psfFluxErr"

# ── Candidate column names for the science-image flux ────────────────────────
# The Fink broker / APDB schema may expose psfScience under different names.
# We will probe the actual parquet columns at load time (Section 3).
SCIENCE_FLUX_CANDIDATES = [
    "r:psfScience",  # most likely Fink-prefixed name
    "r:scienceFlux",
    "r:psfScienceFlux",
    "psfScience",  # without r: prefix
    "scienceFlux",
    "psfScienceFlux",
]
SCIENCE_ERR_CANDIDATES = [
    "r:psfScienceSigma",
    "r:scienceFluxErr",
    "r:psfScienceFluxErr",
    "r:psfScienceErr",
    "psfScienceSigma",
    "scienceFluxErr",
    "psfScienceFluxErr",
]


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load data

In [ ]:
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file: {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file: {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found. "
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Source label: {src_label}")

## 3. Schema inspection — detect psfScience column

We probe the parquet schema to find which column carries the raw science-image
PSF flux.  The DIA alert schema calls it `psfScience` but the Fink broker may
prefix it or rename it slightly.

In [ ]:
# Print all columns for inspection
print("All columns in the loaded table:")
for col in sorted(df.columns):
    print(f"  {col:50s}  dtype={df[col].dtype}")

In [ ]:
# ── Probe for science flux column ─────────────────────────────────────────────
COL_SCI = None
COL_SCIERR = None

for candidate in SCIENCE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_SCI = candidate
        print(f"Found science flux column  : '{COL_SCI}'")
        break

for candidate in SCIENCE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_SCIERR = candidate
        print(f"Found science flux err col : '{COL_SCIERR}'")
        break

if COL_SCI is None:
    print()
    print("WARNING: No psfScience-like column found in the parquet file.")
    print("Columns containing 'science' or 'Science' (case-insensitive):")
    sci_cols = [c for c in df.columns if "science" in c.lower() or "sci" in c.lower()]
    if sci_cols:
        for c in sci_cols:
            print(f"  {c}")
    else:
        print("  (none found)")
    print()
    print("The Fink API /api/v1/objects endpoint does not always return psfScience.")
    print("See Section 3b below for alternative strategies.")
else:
    # Quick sanity check: fraction of non-NaN values
    n_valid = df[COL_SCI].notna().sum()
    print(f"Non-NaN values in '{COL_SCI}': {n_valid:,} / {len(df):,} ({100 * n_valid / len(df):.1f}%)")

### 3b. If psfScience is absent: alternative strategies

If `psfScience` is not present in the Fink alert table, there are several options:

1. **Butler forced photometry on the science calexp** using `lsst.meas.base` or
   the `forcedPhotCcd` task at the positions of the diaObjects.  This gives a
   direct measurement of the science-image flux without going through DIA.

2. **`ssObject` or `diaSource` table from the APDB** — the APDB schema stores
   `psfFlux`, `psfFluxErr`, `psfDiffFlux`, `psfDiffFluxErr`, `psfScienceFlux`,
   and `psfScienceFluxErr`.  If the Butler-joined parquet was written before
   notebook 03 included those columns, the table needs to be regenerated.

3. **Reconstruct from forced-photometry parquet** — if notebook 01 or 03 wrote
   a `gaia_star_stable_src.parquet` that contains the per-visit forced-phot
   columns, those could be used directly.

Check the APDB diaSource schema with:
```python
from lsst.dax.apdb import Apdb
apdb = Apdb.from_uri("...")
print(apdb.schema.diaSources.columns.keys())
```

Or inspect the raw Fink field list with the `/api/v1/columns` endpoint.

In [ ]:
# ── Also look for psfTemplate if available ────────────────────────────────────
TEMPLATE_CANDIDATES = [
    "r:psfTemplate",
    "r:templateFlux",
    "r:psfTemplateFlux",
    "r:psfDiffFlux",  # sometimes template is stored as diff
    "psfTemplate",
    "templateFlux",
]
COL_TMPL = None
for candidate in TEMPLATE_CANDIDATES:
    if candidate in df.columns:
        COL_TMPL = candidate
        print(f"Found template flux column : '{COL_TMPL}'")
        break

if COL_TMPL is None:
    print("No psfTemplate column found — will estimate as psfScience - psfFlux if both are available.")

has_science = COL_SCI is not None
has_sci_err = COL_SCIERR is not None
has_template = COL_TMPL is not None

print(f"\nSummary of available columns:")
print(f"  psfFlux      : {COL_PSF}     (always present)")
print(f"  psfScience   : {COL_SCI}")
print(f"  psfScienceErr: {COL_SCIERR}")
print(f"  psfTemplate  : {COL_TMPL}")

## 4. Rank DIA objects by number of alerts (decreasing)

In [ ]:
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)

print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP} by alert count:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)

print(f"Rows kept for top-{N_TOP} objects : {len(df_top):,}")

## 5. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """Convert MJD array (float) to list of ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_relative_flux(flux, flux_err=None):
    """
    Compute ratio = flux / median(flux) with optional error propagation.

    Returns
    -------
    ratio      : ndarray
    ratio_err  : ndarray or None
    f_med      : float — median flux used for normalisation
    sigma_ratio: float — RMS scatter of ratio
    """
    f = np.asarray(flux, dtype=float)
    finite_mask = np.isfinite(f) & (f > 0)
    if finite_mask.sum() == 0:
        nan_arr = np.full_like(f, np.nan)
        return nan_arr, nan_arr, np.nan, np.nan

    f_med = float(np.median(f[finite_mask]))
    if f_med == 0.0:
        nan_arr = np.full_like(f, np.nan)
        return nan_arr, nan_arr, 0.0, np.nan

    ratio = f / f_med
    if flux_err is not None:
        fe = np.asarray(flux_err, dtype=float)
        ratio_err = fe / f_med
    else:
        ratio_err = None

    sigma_ratio = float(np.nanstd(ratio[finite_mask]))
    return ratio, ratio_err, f_med, sigma_ratio


def _add_date_axis(ax, dt_array, t0_mjd):
    """Add a secondary x-axis on top of *ax* showing calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_mjd = t0_mjd + tick_dt
    tick_dates = mjd_to_datestr(tick_mjd)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


print("Utility functions defined.")

## 6. Loop: psfScience relative flux per band

If `psfScience` is available, we display:
- `psfScience / median(psfScience)` per band (filled circles, band colour)

Same 7-panel layout as notebook 04: `u | g | r | i | z | y | all-bands`.

> **If psfScience is absent**, a diagnostic message is printed and this section is skipped.

In [ ]:
if not has_science:
    print("psfScience column not available — Section 6 skipped.")
    print("See Section 3b for how to add psfScience to the parquet file.")
else:
    n_subplots = len(BANDS) + 1

    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        n_alerts = len(df_obj)
        t0_mjd = df_obj[COL_MJD].min()
        t0_date = mjd_to_datestr([t0_mjd])[0]
        field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""

        fig, axes = plt.subplots(
            1,
            n_subplots,
            figsize=(3.2 * n_subplots, 4.5),
            sharey=False,
            constrained_layout=True,
        )

        # loop on bands
        for idx, band in enumerate(BANDS):
            ax = axes[idx]
            mask = df_obj[COL_BAND] == band
            df_b = df_obj[mask].sort_values(COL_MJD)

            if len(df_b) == 0:
                ax.text(
                    0.5,
                    0.5,
                    "no data",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="gray",
                    fontsize=8,
                )
                ax.set_title(f"band {band}", fontsize=9)
                continue

            dt = df_b[COL_MJD].values - t0_mjd
            color = BAND_COLORS.get(band, "k")

            sci_err_vals = df_b[COL_SCIERR].values if has_sci_err else None
            ratio, ratio_err, f_med, sigma = compute_relative_flux(df_b[COL_SCI].values, sci_err_vals)

            ax.errorbar(
                dt,
                ratio,
                yerr=ratio_err if ratio_err is not None else None,
                fmt="o",
                ms=4,
                color=color,
                ecolor=color,
                elinewidth=0.8,
                capsize=2,
                alpha=0.85,
                label=f"{band}  σ={sigma:.3f}",
            )
            ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
            ax.set_title(f"band {band}  |  σ={sigma:.3f}", fontsize=9)
            ax.set_ylabel("psfScience / median", fontsize=8)
            ax.set_xlabel("Δt [days]", fontsize=8)
            _add_date_axis(ax, dt, t0_mjd)

        # Combined panel
        ax_all = axes[-1]
        for band in BANDS:
            mask = df_obj[COL_BAND] == band
            df_b = df_obj[mask].sort_values(COL_MJD)
            if len(df_b) == 0:
                continue
            dt = df_b[COL_MJD].values - t0_mjd
            color = BAND_COLORS.get(band, "k")
            sci_err_vals = df_b[COL_SCIERR].values if has_sci_err else None
            ratio, ratio_err, _, sigma = compute_relative_flux(df_b[COL_SCI].values, sci_err_vals)
            ax_all.errorbar(
                dt,
                ratio,
                yerr=ratio_err,
                fmt="o",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.7,
                label=f"{band} σ={sigma:.3f}",
            )
        ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax_all.set_title("all bands", fontsize=9)
        ax_all.set_ylabel("psfScience / median", fontsize=8)
        ax_all.set_xlabel("Δt [days]", fontsize=8)
        ax_all.legend(fontsize=7, ncol=3, loc="best")
        _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

        fig.suptitle(
            f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
            f"  N={n_alerts}  |  t₀={t0_date}\n[psfScience / median]",
            fontsize=11,
            y=1.06,
        )
        savefig(f"sci_relflux_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
        plt.show()

## 7. Loop: overlay psfScience/median vs psfFlux/median

This is the key diagnostic plot:

- **Filled circles** = `psfScience / median(psfScience)` — raw science flux
- **Open squares**  = `psfFlux    / median(psfFlux)`    — difference-image flux

If the two curves are consistent, the template is well-subtracted and stable.
If `psfFlux` shows excess variability absent from `psfScience`, the template
is the culprit.  Conversely, if both vary similarly, the variability is real
(atmospheric, instrumental gain, etc.).

In [ ]:
if not has_science:
    print("psfScience column not available — Section 7 skipped.")
else:
    n_subplots = len(BANDS) + 1

    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        n_alerts = len(df_obj)
        t0_mjd = df_obj[COL_MJD].min()
        t0_date = mjd_to_datestr([t0_mjd])[0]
        field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""

        fig, axes = plt.subplots(
            1,
            n_subplots,
            figsize=(3.2 * n_subplots, 4.5),
            sharey=False,
            constrained_layout=True,
        )

        for idx, band in enumerate(BANDS):
            ax = axes[idx]
            mask = df_obj[COL_BAND] == band
            df_b = df_obj[mask].sort_values(COL_MJD)

            if len(df_b) == 0:
                ax.text(
                    0.5,
                    0.5,
                    "no data",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="gray",
                    fontsize=8,
                )
                ax.set_title(f"band {band}", fontsize=9)
                continue

            dt = df_b[COL_MJD].values - t0_mjd
            color = BAND_COLORS.get(band, "k")

            # ── psfScience / median ────────────────────────────────────────────
            sci_err_vals = df_b[COL_SCIERR].values if has_sci_err else None
            sci_ratio, sci_err, _, sigma_sci = compute_relative_flux(df_b[COL_SCI].values, sci_err_vals)
            ax.errorbar(
                dt,
                sci_ratio,
                yerr=sci_err,
                fmt="o",
                ms=5,
                color=color,
                ecolor=color,
                elinewidth=0.9,
                capsize=2,
                alpha=0.90,
                label=f"sci  σ={sigma_sci:.3f}",
            )

            # ── psfFlux / median ───────────────────────────────────────────────
            psf_ratio, psf_err, _, sigma_psf = compute_relative_flux(
                df_b[COL_PSF].values, df_b[COL_PSFERR].values
            )
            ax.errorbar(
                dt,
                psf_ratio,
                yerr=psf_err,
                fmt="s",
                ms=4,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.55,
                mfc="none",
                label=f"diff σ={sigma_psf:.3f}",
            )

            ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
            ax.set_title(f"{band}  sci σ={sigma_sci:.3f}  diff σ={sigma_psf:.3f}", fontsize=8)
            ax.set_ylabel("flux / median", fontsize=8)
            ax.set_xlabel("Δt [days]", fontsize=8)
            ax.legend(fontsize=7, loc="best")
            _add_date_axis(ax, dt, t0_mjd)

        # Combined panel
        ax_all = axes[-1]
        for band in BANDS:
            mask = df_obj[COL_BAND] == band
            df_b = df_obj[mask].sort_values(COL_MJD)
            if len(df_b) == 0:
                continue
            dt = df_b[COL_MJD].values - t0_mjd
            color = BAND_COLORS.get(band, "k")

            sci_err_vals = df_b[COL_SCIERR].values if has_sci_err else None
            sci_ratio, sci_err, _, _ = compute_relative_flux(df_b[COL_SCI].values, sci_err_vals)
            psf_ratio, psf_err, _, _ = compute_relative_flux(df_b[COL_PSF].values, df_b[COL_PSFERR].values)

            ax_all.errorbar(
                dt,
                sci_ratio,
                yerr=sci_err,
                fmt="o",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.6,
                capsize=2,
                alpha=0.80,
                label=f"{band} sci",
            )
            ax_all.errorbar(
                dt,
                psf_ratio,
                yerr=psf_err,
                fmt="s",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.6,
                capsize=2,
                alpha=0.45,
                mfc="none",
                label=f"{band} diff",
            )

        ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax_all.set_title("all bands — sci (●) vs diff (□)", fontsize=9)
        ax_all.set_ylabel("flux / median", fontsize=8)
        ax_all.set_xlabel("Δt [days]", fontsize=8)
        ax_all.legend(fontsize=6, ncol=4, loc="best")
        _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

        fig.suptitle(
            f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
            f"  N={n_alerts}  |  t₀={t0_date}\n"
            "psfScience/median (●) vs psfFlux/median (□)",
            fontsize=11,
            y=1.07,
        )
        savefig(f"sci_vs_diff_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
        plt.show()

## 8. Estimated template variability

We reconstruct the per-epoch template flux as:

```
psfTemplate_est(t) = psfScience(t) - psfFlux(t)
```

For a stable star observed with a stable template, this quantity should be
**constant** (equal to the static template depth).  Temporal variations in
`psfTemplate_est` indicate either:
- noise in the template co-add propagated per epoch (correlated with seeing / sky),
- a per-visit photometric calibration drift in the science image,
- or detector-level (CCD-dependent) flat-fielding residuals.

We plot `psfTemplate_est / median(psfTemplate_est)` per band.

In [ ]:
if not has_science:
    print("psfScience column not available — Section 8 skipped.")
else:
    n_subplots = len(BANDS) + 1

    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD).copy()
        n_alerts = len(df_obj)
        t0_mjd = df_obj[COL_MJD].min()
        t0_date = mjd_to_datestr([t0_mjd])[0]
        field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""

        # Compute estimated template flux
        df_obj["psfTemplate_est"] = df_obj[COL_SCI].values - df_obj[COL_PSF].values

        fig, axes = plt.subplots(
            1,
            n_subplots,
            figsize=(3.2 * n_subplots, 4.5),
            sharey=False,
            constrained_layout=True,
        )

        for idx, band in enumerate(BANDS):
            ax = axes[idx]
            mask = df_obj[COL_BAND] == band
            df_b = df_obj[mask].sort_values(COL_MJD)

            if len(df_b) == 0:
                ax.text(
                    0.5,
                    0.5,
                    "no data",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="gray",
                    fontsize=8,
                )
                ax.set_title(f"band {band}", fontsize=9)
                continue

            dt = df_b[COL_MJD].values - t0_mjd
            color = BAND_COLORS.get(band, "k")

            tmpl_vals = df_b["psfTemplate_est"].values
            ratio, ratio_err, f_med, sigma = compute_relative_flux(tmpl_vals)

            ax.plot(
                dt,
                ratio,
                "D",
                ms=4,
                color=color,
                alpha=0.80,
                label=f"{band}  σ={sigma:.3f}",
            )
            ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
            ax.set_title(f"band {band}  |  σ={sigma:.3f}", fontsize=9)
            ax.set_ylabel("psfTemplate_est / median", fontsize=8)
            ax.set_xlabel("Δt [days]", fontsize=8)
            _add_date_axis(ax, dt, t0_mjd)

        # Combined panel
        ax_all = axes[-1]
        for band in BANDS:
            mask = df_obj[COL_BAND] == band
            df_b = df_obj[mask].sort_values(COL_MJD)
            if len(df_b) == 0:
                continue
            dt = df_b[COL_MJD].values - t0_mjd
            color = BAND_COLORS.get(band, "k")
            ratio, _, _, sigma = compute_relative_flux(df_b["psfTemplate_est"].values)
            ax_all.plot(dt, ratio, "D", ms=3, color=color, alpha=0.7, label=f"{band} σ={sigma:.3f}")

        ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax_all.set_title("all bands — estimated template", fontsize=9)
        ax_all.set_ylabel("psfTemplate_est / median", fontsize=8)
        ax_all.set_xlabel("Δt [days]", fontsize=8)
        ax_all.legend(fontsize=7, ncol=3, loc="best")
        _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

        fig.suptitle(
            f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
            f"  N={n_alerts}  |  t₀={t0_date}\n"
            "Estimated template psfTemplate_est = psfScience − psfFlux  /  median",
            fontsize=11,
            y=1.07,
        )
        savefig(f"template_est_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
        plt.show()

## 9. Summary table: σ(ratio) for psfScience vs psfFlux

Tabulate the RMS scatter of the normalised science flux and the normalised
difference flux, per object and per band.  A direct comparison shows whether
the variability is amplified by the template subtraction.

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].copy()
    if has_science:
        df_obj["psfTemplate_est"] = df_obj[COL_SCI].values - df_obj[COL_PSF].values
    n_total = len(df_obj)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_alerts": n_total,
        "t0_date": t0_date,
    }

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"sigma_sci_{band}"] = np.nan
            row[f"sigma_psf_{band}"] = np.nan
            row[f"sigma_tmpl_{band}"] = np.nan
            continue

        if has_science:
            sci_err_vals = df_b[COL_SCIERR].values if has_sci_err else None
            _, _, _, sigma_sci = compute_relative_flux(df_b[COL_SCI].values, sci_err_vals)
            _, _, _, sigma_tmpl = compute_relative_flux(df_b["psfTemplate_est"].values)
            row[f"sigma_sci_{band}"] = round(sigma_sci, 4)
            row[f"sigma_tmpl_{band}"] = round(sigma_tmpl, 4)
        else:
            row[f"sigma_sci_{band}"] = np.nan
            row[f"sigma_tmpl_{band}"] = np.nan

        _, _, _, sigma_psf = compute_relative_flux(df_b[COL_PSF].values, df_b[COL_PSFERR].values)
        row[f"sigma_psf_{band}"] = round(sigma_psf, 4)

    records.append(row)

df_summary = pd.DataFrame(records)
print("Summary: σ(relative flux) per object and band")
print("  sci  = psfScience / median  (raw science image flux)")
print("  psf  = psfFlux    / median  (difference-image flux)")
print("  tmpl = psfTemplate_est / median  (= psfScience - psfFlux)")
print()
display(df_summary)

In [ ]:
# Save summary CSV
summary_path = os.path.join(DIR_FIGS, f"sigma_sci_vs_psf_summary_{src_label}.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved → {summary_path}")

## 10. Interpretation guide

| Observation | Interpretation |
|---|---|
| σ(sci) ≈ σ(psf) | Template is stable; variability is real (atmospheric or instrumental gain) |
| σ(psf) > σ(sci) | Template subtracts too much or too little → template is the problem |
| σ(tmpl) >> 0    | Template flux is not constant epoch-to-epoch → template noise or drift |
| σ(sci) varies across diaObjects at same epoch | Detector-level (CCD) flat-field or gain residuals |
| σ(sci) correlated with airmass / seeing | Atmospheric origin (PWV, clouds, differential chromatic refraction) |
| σ(sci) correlated across bands but same CCD | Common-mode gain drift (voltage, temperature) |

**Next steps if template is the problem:**
- Check the visit MJD used to build the template (column `r:midpointMjdTai` of the template visit if available)
- Look at per-CCD templates vs. global template depth
- Consider regenerating the template from a different epoch range using the Butler

**Next steps if psfScience itself varies:**
- Cross-match with `visit_summary_src.csv` for seeing, sky background, photometric zero-point
- Compare with PWV measurements at the LSST site (AuxTel) for atmospheric origin
- Plot vs. CCD detector number to separate spatial from temporal effects